# ML-03 — Frame Your Lane as an ML Task

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/bhardwaj-ayush03/flyrank-internship/blob/main/work/notebooks/w02_ml_task_framing.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My lane as an ML task (type)

**Lane:** Refresh / Content Opportunity Scoring (provisional).

**Task type: ranking, built on a classification score.** A model gives each page a probability of being in the "declining" group, and I sort pages by that score to make a review queue. What matters is the order of the top of the list, not a yes/no for every page.

**ML loop:** question (which pages should be reviewed first?) -> data (one row per page from the starter CSV) -> baseline (hand rule: stale x visible) -> model (decision tree, then stronger models) -> evaluation on held-out clients -> action (review queue with reason codes).

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 2. Target or proxy

**Proxy target:** `is_declining_label = (trend_direction == "down")`, 1 if the page's recent trend is down, else 0.

This is a proxy, not the real goal. It is computed from the current window, not from a future outcome, so it says "was declining" rather than "will decline". A stronger version for later would use features from a prior window and a decline measured over the next window. `trend_pct` must never be a feature, because the label is derived from it (leakage).

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 3. Success metric

**Main metric: Precision@50** on held-out clients: of the 50 pages the system ranks highest, what fraction are labelled declining. I match K to review capacity (a reviewer can check about 20-50 pages per cycle), and I also report Precision@20.

**Reference points:** the base rate is 0.542 (16,262 of 30,000 pages), so a ranking must beat about 0.54 to show any skill, and it must beat the hand rule on the same held-out clients. I will also read my own top 20 pages by hand. Split by `client_id` (client-holdout), never randomly.

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 4. The unit of analysis, as a real dataframe

**One row = one page** (one `content_id`, belonging to one `client_id`). Filters follow the starter pipeline: impressions > 0, page at least 90 days old, one row per page.

In [1]:
from pathlib import Path
import subprocess
import pandas as pd

CSV = "data/raw/content_refresh_anonymized.csv"
root = next((Path(p) for p in [".", "..", "../.."] if (Path(p) / CSV).exists()), None)
if root is None:  # Colab opened straight from GitHub: pull the public starter data
    subprocess.run(["git", "clone", "--depth", "1",
                    "https://github.com/flyrank-bih/flyrank-ml-internship-starter", "_starter"], check=True)
    root = Path("_starter")
raw = pd.read_csv(root / CSV)

pages = (raw[(raw["impressions_90d"] > 0) & (raw["content_age_days"] >= 90)]
         .drop_duplicates("content_id").copy())
pages["is_declining_label"] = pages["trend_direction"].str.lower().eq("down").astype(int)

cols = ["content_id", "client_id", "impressions_90d", "avg_position", "ctr",
        "content_age_days", "days_since_last_update", "word_count",
        "trend_direction", "is_declining_label"]
print("rows (pages):", len(pages), "| clients:", pages["client_id"].nunique())
print("pages per client (median):", int(pages.groupby("client_id").size().median()))
print(pages["is_declining_label"].value_counts().rename({1: "declining", 0: "not declining"}))
pages[cols].head(5)

rows (pages): 30000 | clients: 32
pages per client (median): 567
is_declining_label
declining        16262
not declining    13738
Name: count, dtype: int64


,content_id,client_id,impressions_90d,avg_position,ctr,content_age_days,days_since_last_update,word_count,trend_direction,is_declining_label
0,content_304f48230142,client_f369cb89fc,3803,10.6,0.76,187,20,3221.0,down,1
1,content_a1fb4e703a9e,client_4e07408562,15320,20.3,0.05,445,25,2481.0,down,1
2,content_9aa793d4d895,client_7f2253d7e2,12581,36.5,0.09,141,20,3515.0,down,1
3,content_331d6c4de07b,client_19581e27de,11751,6.2,0.49,463,22,NaN,stable,0
4,content_d99b7a2d90ca,client_3fdba35f04,19140,44.0,0.13,263,14,2803.0,down,1


**Sketch of the target column:** `is_declining_label` is 1 for 16,262 pages (54.2%) and 0 for 13,738. The features would come from columns like age, visibility, position, CTR and word count. `trend_direction` and `trend_pct` are excluded from features because they define the label.

## 5. Why ML beats a fixed rule here

My hand rule (stale >= 180 days and visible >= 500 impressions) matches only 17 pages out of 30,000, so most of a top-50 list is tied scores. In the same sample, stale pages decline slightly less often (47.1%) than non-stale ones (54.2%), so one-signal intuition is unreliable. The signal is spread across several columns (age, visibility, position, CTR, depth), and a model can weigh them together and score every page.

**This is a hypothesis, not a result.** In my Week-2 experiment on held-out clients the shallow trees did not clearly beat the hand rule (Precision@50 about 0.52-0.56 vs 0.64), so I will test stronger models and a fairer validation before claiming ML is better.

**Action it supports:** the ranked queue tells a content owner which pages to review first, and each page gets a reason code (for example stale but visible, or declining with demand) so a person can decide whether to refresh, protect or monitor it.

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.